<a href="https://colab.research.google.com/github/Emmanuel-NS/FARUMASI-AI/blob/main/notebooks/train_chat_reader_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chat reader: Qwen2.5-1.5B-Instruct + LoRA

Runtime → Change runtime type → **T4 GPU**. Run the cells in order (about 1.5–2 hours in total).

Fine-tunes Qwen2.5-1.5B-Instruct with LoRA to write pharmacist-style answers grounded on the retrieved passages (direct first sentence, then details, and "my sources don't cover this" when the passages do not answer). Then compares it with the zero-shot model on the same test questions as the Flan-T5 reader.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "https://github.com/Emmanuel-NS/FARUMASI-AI.git"
%cd /content
!rm -rf repo && git clone -q $REPO_URL repo
%cd /content/repo
!pip install -q -r requirements.txt

name, memory.total [MiB]
Tesla T4, 15360 MiB
/content
/content/repo
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 430.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.6/325.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

## 1. Data (same corpus and splits as the main notebook)

In [2]:
!python -m pharmqa.data.download
!python -m pharmqa.data.prepare

MedQuAD: 11274 XML documents -> /content/repo/data/raw/medquad
README.md: 100% 541/541 [00:00<00:00, 2.27MB/s]

data/train-00000-of-00001-7427a10e891759(…): downloading bytes:   0% 0.00/215k [00:00<?, ?B/s]
data/train-00000-of-00001-7427a10e891759(…): downloading bytes: 100% 213k/213k [00:01<00:00, 182kB/s, 20.7kB/s  ]
data/train-00000-of-00001-7427a10e891759(…): reconstructing file: 100% 215k/215k [00:01<00:00, 183kB/s, 20.8kB/s  ]
Generating train split: 100% 690/690 [00:00<00:00, 18880.32 examples/s]
Creating json from Arrow format: 100% 1/1 [00:00<00:00, 60.12ba/s]
MedicationQA: 690 rows
README.md: 100% 24.0k/24.0k [00:00<00:00, 59.8MB/s]

plus/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/312k [00:00<?, ?B/s]
plus/train-00000-of-00001.parquet: downloading bytes: 100% 296k/296k [00:01<00:00, 290kB/s, 28.8kB/s  ]
plus/train-00000-of-00001.parquet: reconstructing file: 100% 312k/312k [00:01<00:00, 306kB/s, 30.4kB/s  ]

plus/validation-00000-of-00001.parquet: downloading 

## 2. LoRA fine-tuning (about 60–80 min on a T4)

3,000 answer examples + up to 400 multi-section overviews + about 8% "no answer" examples, 1 epoch. If Colab disconnects, re-run the cell with `--resume` added.

In [3]:
!python -m pharmqa.generation.train_chat_lora --epochs 1 --max_train 3000 --batch 2 --grad_accum 8 --resume

config.json: 100% 660/660 [00:00<00:00, 3.64MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 20.8MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 117MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 117MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 126MB/s]
answers (train): 100% 3000/3000 [00:25<00:00, 118.80it/s]
3251 examples (36 longer than 1536 tokens skipped)
answers (val): 100% 200/200 [00:01<00:00, 103.26it/s]
219 examples (6 longer than 1536 tokens skipped)

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   3% 96.3M/3.09G [00:02<00:26, 115MB/s, 5.81MB/s  ] 
model.safetensors: downloading bytes:   4% 137M/3.09G [00:02<00:20, 144MB/s, 10.7MB/s  ] 
model.safetensors: downloading bytes:   5% 164M/3.09G [00:02<00:21, 139MB/s, 13.6MB/s  ]
model.safetensors: downloading bytes:   7% 217M/3.09G [00:03<00:21, 136MB/s, 17.0MB/s  ]
model.safetensors: downloading bytes:   8% 242M/3.09G [00:03<00:18, 156MB/s, 19.2MB/

## 3. Evaluation (about 20–30 min)

Gold context = the right passage + 2 hard negatives (same condition as the Flan-T5 table). Retrieved context falls back to BM25 here because the fine-tuned retriever is not trained in this notebook.

In [4]:
for system in ["zeroshot", "lora"]:
    !python -m pharmqa.generation.evaluate_chat --system {system} --context gold --n 200

hard negatives: 100% 199/199 [00:01<00:00, 118.57it/s]
Loading weights: 100% 338/338 [00:06<00:00, 52.27it/s]
100% 199/199 [00:00<00:00, 4667.79it/s]
unrelated: 100% 60/60 [00:00<00:00, 117.34it/s]
config.json: 100% 483/483 [00:00<00:00, 2.06MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 184kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 3.17MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 2.98MB/s]

model.safetensors: downloading bytes:  86% 231M/268M [00:01<00:00, 250MB/s, 18.4MB/s  ]
model.safetensors: reconstructing file:  84% 226M/268M [00:03<00:00, 72.0MB/s]
model.safetensors: downloading bytes: 100% 250M/250M [00:03<00:00, 76.6MB/s, 23.2MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:03<00:00, 82.0MB/s, 24.4MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 11643.41it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias 

In [5]:
!python -m pharmqa.generation.evaluate_chat --system lora --context retrieved --n 200 --tag _bm25

BM25: 100% 199/199 [00:01<00:00, 118.65it/s]
Loading weights: 100% 338/338 [00:02<00:00, 127.78it/s]
100% 199/199 [00:00<00:00, 4713.66it/s]
unrelated: 100% 60/60 [00:00<00:00, 117.28it/s]
Loading weights: 100% 100/100 [00:00<00:00, 5884.93it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
{
  "system": "qwen2.5-1.5b_lora_bm25",
  "context": "retrieved",
  "n": 199,
  "rougeL": 0.4322,
  "rougeL_recall": 0.6072,
  "bertscore_f1": 0.8468,
  "support": 0.9298,
  "avg_words": 98.6,
  "false_abstain": 0.0,
  "abstain_on_unrelated": 0.7333,
  "by_so

## 4. Download the adapter and results

Unzip `chat_reader.zip` into the project folder on your computer (it creates `artifacts/generator_qwen_lora/` and `results/chat_*`).

In [6]:
!rm -rf artifacts/generator_qwen_lora/checkpoints
!zip -qr chat_reader.zip artifacts/generator_qwen_lora results/chat_*
from google.colab import files
files.download("chat_reader.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>